Phase 1 → Dataset Understanding

Phase 2 → Data Cleaning

Phase 3 → EDA

Phase 4 → Visualization

Phase 5 → Insight Generation

Phase 6 → Report Generation

Phase 7 → Natural Language Querying

Phase 8 → Autonomous Agentic System

# Phase 1: Dataset Understanding Agent
The first responsibility of our AI agent is to understand the dataset automatically.

The agent should answer:

How many rows and columns?

What are the column names?

What is the datatype of each column?

Which columns are numerical?

Which columns are categorical?

Which columns contain dates?

Which columns contain IDs?

# Create the loader Agent

In [1]:
import pandas as pd

class LoaderAgent:

    def load_data(self, file_path):

        if file_path.endswith(".csv"):
            df = pd.read_csv(file_path)

        elif file_path.endswith(".xlsx"):
            df = pd.read_excel(file_path)

        else:
            raise ValueError("Unsupported file format")

        print("✅ Dataset Loaded")

        return df

# Build Dataset Understanding Agent

In [2]:
class DatasetUnderstandingAgent:
    def analyze(self,df):
        report={}
        report["rows"]=df.shape[0]
        report["columns"]=df.shape[1]
        report["column_names"]=list(df.columns)
        report["datatypes"]=df.dtypes.astype(str).to_dict()
        report["numerical_columns"]=list(df.select_dtypes(include="number").columns)
        report["categorical_columns"]=list(df.select_dtypes(include="object").columns)
        return report    

# Test the Agents

In [3]:
loader=LoaderAgent()
df=loader.load_data("employees.csv")
understanding_agent=DatasetUnderstandingAgent()
report=understanding_agent.analyze(df)

✅ Dataset Loaded


# Display Report Nicely

In [4]:
for key,value in report.items():
    print(f"🔹{key.upper()}:{value}")
    

🔹ROWS:7
🔹COLUMNS:4
🔹COLUMN_NAMES:['Name', 'Department', 'Salary', 'Experience']
🔹DATATYPES:{'Name': 'object', 'Department': 'object', 'Salary': 'float64', 'Experience': 'int64'}
🔹NUMERICAL_COLUMNS:['Salary', 'Experience']
🔹CATEGORICAL_COLUMNS:['Name', 'Department']


# Phase 2: Data Cleaning Agent

This agent will automatically:

✅ Detect missing values

✅ Fill missing values intelligently

✅ Remove duplicates

✅ Fix incorrect datatypes

✅ Detect outliers

✅ Generate a cleaning report

# Create the Cleaner Agent

In [5]:
import pandas as pd
import numpy as np

class CleanerAgent:

    def clean_data(self, df):

        report = {}

        # =====================
        # Missing Values
        # =====================

        missing_before = df.isnull().sum()

        report["missing_before"] = missing_before.to_dict()

        # Numerical columns
        num_cols = df.select_dtypes(include=np.number).columns

        for col in num_cols:
            df[col] = df[col].fillna(df[col].median())

        # Categorical columns
        cat_cols = df.select_dtypes(include="object").columns

        for col in cat_cols:
            df[col] = df[col].fillna(df[col].mode()[0])

        # =====================
        # Duplicates
        # =====================

        duplicates = df.duplicated().sum()

        report["duplicates_removed"] = duplicates

        df = df.drop_duplicates()

        # =====================
        # Missing After Cleaning
        # =====================

        report["missing_after"] = (
            df.isnull().sum().to_dict()
        )

        print("✅ Data Cleaning Completed")

        return df, report

# Run the Cleaner Agent


In [6]:
cleaner=CleanerAgent()

cleaned_df,cleaning_report=cleaner.clean_data(df)


✅ Data Cleaning Completed


# View the Cleaning Report

In [7]:
for key,value in cleaning_report.items():
    print("\n","="*50)
    print(key.upper())
    print("="*50)
    print(value)


MISSING_BEFORE
{'Name': 0, 'Department': 0, 'Salary': 1, 'Experience': 0}

DUPLICATES_REMOVED
0

MISSING_AFTER
{'Name': 0, 'Department': 0, 'Salary': 0, 'Experience': 0}


# Add Outlier Detection

We'll use the IQR Method

In [8]:
class OutlierAgent:
    def detect_outliers(self,df):
        outlier_report={}
        numeric_cols=df.select_dtypes(include=np.number).columns
        for col in numeric_cols:
            Q1=df[col].quantile(0.25)
            Q3=df[col].quantile(0.75)
            
            IQR=Q3-Q1

            lower=Q1-1.5*IQR
            upper=Q3+1.5*IQR
            outliers=df[(df[col]<lower)|(df[col]>upper)]
            outlier_report[col]=len(outliers)
        return outlier_report
        

# Run It

In [9]:
outlier_agent=OutlierAgent()
outliers=outlier_agent.detect_outliers(cleaned_df)
print(outliers)

{'Salary': 0, 'Experience': 0}


In [10]:
print(cleaned_df.shape)
print(cleaned_df.head())

(7, 4)
    Name Department   Salary  Experience
0    Ram         IT  50000.0           2
1   Sita         HR  60000.0           5
2   John         IT  70000.0           4
3  David    Finance  80000.0           7
4   Sara         HR  55000.0           3


# Phase 3: EDA Agent

This agent will automatically generate:

Summary statistics

Correlation matrix

Feature distributions

Skewness analysis

Dataset health score


# Create EDA Agent

In [11]:
import numpy as np
class EDAAgent:
    def analyze(self,df):
        report={}
        #Dataset Shape
        report["shape"]=df.shape
        #Column Names
        report["columns"]=list(df.columns)
        #Data types
        report["dtypes"]=df.dtypes.astype(str).to_dict()
        #Summary Statistics
        report["summary_statistics"]=(df.describe(include='all').to_dict())
        #Correlation
        numeric_df=df.select_dtypes(include=np.number)
        if not numeric_df.empty:
            report["correlation"]=(numeric_df.corr().to_dict())
        #Unique Values
        unique_values={}
        for col in df.columns:
            unique_values[col]=df[col].nunique()
        report["unique_values"]=unique_values
        #Skewness
        skewness={}
        for col in numeric_df.columns:
            skewness[col]=round(df[col].skew(),2)
        report["skewness"]=skewness

        print("✅ EDA Completed")
        return report

# Run EDA Agent

In [12]:
eda_agent=EDAAgent()
eda_report=eda_agent.analyze(cleaned_df)

✅ EDA Completed


# View Report
Instead of printing everything at once

In [13]:
for key in eda_report:
    print("\n" + "="*50)
    print(key.upper())
    print("="*50)

    print(eda_report[key])


SHAPE
(7, 4)

COLUMNS
['Name', 'Department', 'Salary', 'Experience']

DTYPES
{'Name': 'object', 'Department': 'object', 'Salary': 'float64', 'Experience': 'int64'}

SUMMARY_STATISTICS
{'Name': {'count': 7, 'unique': 7, 'top': 'Ram', 'freq': 1, 'mean': nan, 'std': nan, 'min': nan, '25%': nan, '50%': nan, '75%': nan, 'max': nan}, 'Department': {'count': 7, 'unique': 3, 'top': 'IT', 'freq': 3, 'mean': nan, 'std': nan, 'min': nan, '25%': nan, '50%': nan, '75%': nan, 'max': nan}, 'Salary': {'count': 7.0, 'unique': nan, 'top': nan, 'freq': nan, 'mean': 67142.85714285714, 'std': 14099.98311380279, 'min': 50000.0, '25%': 57500.0, '50%': 65000.0, '75%': 75000.0, 'max': 90000.0}, 'Experience': {'count': 7.0, 'unique': nan, 'top': nan, 'freq': nan, 'mean': 4.428571428571429, 'std': 2.370453040886408, 'min': 2.0, '25%': 2.5, '50%': 4.0, '75%': 6.0, 'max': 8.0}}

CORRELATION
{'Salary': {'Salary': 1.0, 'Experience': 0.8655198623323144}, 'Experience': {'Salary': 0.8655198623323144, 'Experience': 1.0

# Phase 4: Visualiation Agent
Now the AI will automatically create charts for every colums.

# Import libraries

In [14]:
import matplotlib.pyplot as plt
import seaborn as sns
plt.style.use('ggplot')

# Create Visualization Agent

In [15]:
import os
class VisualizationAgent:
    def create_visualization(self,df):
        os.makedirs("charts",exist_ok=True)
        
        numeric_cols=df.select_dtypes(include=np.number).columns
        
        categorical_cols=df.select_dtypes(include='object').columns
        
        # Histograms
        for col in numeric_cols:
            plt.figure(figsize=(6,4))
            sns.histplot(df[col],kde=True)
            
            plt.title(f"Distribution of {col}")
            
            plt.tight_layout()
            
            plt.savefig(f"charts/{col}_histogram.png")
            
            plt.close()
            
        #Boxplots
        for col in numeric_cols:

            plt.figure(figsize=(6,4))

            sns.boxplot(x=df[col])

            plt.title(f"Boxplot of {col}")

            plt.tight_layout()

            plt.savefig(
                f"charts/{col}_boxplot.png"
            )

            plt.close()
        # Countplots
        for col in categorical_cols:

            plt.figure(figsize=(8,4))

            sns.countplot(
                y=df[col],
                order=df[col].value_counts().index
            )

            plt.title(f"{col} Count")

            plt.tight_layout()

            plt.savefig(
                f"charts/{col}_countplot.png"
            )

            plt.close()
        # Correlation Heatmap

        if len(numeric_cols) > 1:

            plt.figure(figsize=(8,6))

            sns.heatmap(
                df[numeric_cols].corr(),
                annot=True,
                cmap="coolwarm"
            )

            plt.title("Correlation Matrix")

            plt.tight_layout()

            plt.savefig(
                "charts/correlation_heatmap.png"
            )

            plt.close()

        print("✅ Visualizations Created")
        

# Run It


In [16]:
visual_agent=VisualizationAgent()
visual_agent.create_visualization(cleaned_df)

✅ Visualizations Created


# Phase 4: Build the Insight Agent

In [17]:
class InsightAgent:

    def generate_insights(self, df):

        insights = []

        numeric_cols = df.select_dtypes(include='number').columns
        categorical_cols = df.select_dtypes(include='object').columns

        # Numerical insights
        for col in numeric_cols:

            insights.append(
                f"Average {col}: {round(df[col].mean(),2)}"
            )

            insights.append(
                f"Maximum {col}: {df[col].max()}"
            )

            insights.append(
                f"Minimum {col}: {df[col].min()}"
            )

        # Categorical insights
        for col in categorical_cols:

            top_category = df[col].mode()[0]

            insights.append(
                f"Most frequent {col}: {top_category}"
            )

        return insights

In [18]:
insight_agent = InsightAgent()

insights = insight_agent.generate_insights(cleaned_df)

for insight in insights:
    print("✅", insight)

✅ Average Salary: 67142.86
✅ Maximum Salary: 90000.0
✅ Minimum Salary: 50000.0
✅ Average Experience: 4.43
✅ Maximum Experience: 8
✅ Minimum Experience: 2
✅ Most frequent Name: David
✅ Most frequent Department: IT


# Phase 6: Report Generator Agent

# Create ReportGeneratorAgent

In [19]:
class ReportGeneratorAgent:

    def generate_report(
            self,
            understanding_report,
            cleaning_report,
            outlier_report,
            insights
    ):

        with open("AI_Analysis_Report.txt", "w",
                  encoding="utf-8") as file:

            file.write("="*60 + "\n")
            file.write("AI DATA ANALYST REPORT\n")
            file.write("="*60 + "\n\n")

            # Dataset Info
            file.write("DATASET INFORMATION\n")
            file.write("-"*40 + "\n")

            file.write(
                f"Rows: {understanding_report['rows']}\n"
            )

            file.write(
                f"Columns: {understanding_report['columns']}\n\n"
            )

            file.write(
                f"Column Names:\n"
            )

            for col in understanding_report["column_names"]:
                file.write(f"- {col}\n")

            # Cleaning Report

            file.write("\n\n")
            file.write("CLEANING REPORT\n")
            file.write("-"*40 + "\n")

            file.write(
                f"Duplicates Removed: "
                f"{cleaning_report['duplicates_removed']}\n\n"
            )

            file.write("Missing Values Before:\n")

            for col, val in cleaning_report[
                    "missing_before"].items():

                file.write(f"{col}: {val}\n")

            file.write("\nMissing Values After:\n")

            for col, val in cleaning_report[
                    "missing_after"].items():

                file.write(f"{col}: {val}\n")

            # Outliers

            file.write("\n\n")
            file.write("OUTLIER REPORT\n")
            file.write("-"*40 + "\n")

            for col, count in outlier_report.items():

                file.write(
                    f"{col}: {count} outliers\n"
                )

            # Insights

            file.write("\n\n")
            file.write("BUSINESS INSIGHTS\n")
            file.write("-"*40 + "\n")

            for insight in insights:

                file.write(f"• {insight}\n")

        print("✅ Report Generated Successfully")

# Phase 7: Natural Language Query Agent

# Create Query Agent


In [20]:
class QueryAgent:

    def answer(self, question, df):

        question = question.lower()

        # Head
        if "head" in question or "top 5" in question:
            return df.head()

        # Shape
        elif "shape" in question:
            return f"Dataset Shape: {df.shape}"

        # Average Salary
        elif "average salary" in question:
            return f"Average Salary: {df['Salary'].mean():.2f}"

        # Maximum Salary
        elif "maximum salary" in question or "highest salary" in question:
            return f"Maximum Salary: {df['Salary'].max()}"

        # Minimum Salary
        elif "minimum salary" in question or "lowest salary" in question:
            return f"Minimum Salary: {df['Salary'].min()}"

        # Department Counts
        elif "department count" in question:
            return df['Department'].value_counts()

        # Missing Values
        elif "missing values" in question:
            return df.isnull().sum()

        else:
            return "Sorry, I don't understand the question."

In [21]:
query_agent = QueryAgent()

In [22]:
query_agent = QueryAgent()

while True:

    question = input("\nAsk your question: ")

    if question.lower() == "exit":
        print("👋 Goodbye")
        break

    response = query_agent.answer(
        question,
        cleaned_df
    )

    print("\nAnswer:")
    print(response)

👋 Goodbye


# Master AI Analyst Agent

In [23]:
class AIAnalystAgent:

    def __init__(self):

        self.loader = LoaderAgent()
        self.understanding = DatasetUnderstandingAgent()
        self.cleaner = CleanerAgent()
        self.outlier = OutlierAgent()
        self.eda = EDAAgent()
        self.visualizer = VisualizationAgent()
        self.insight = InsightAgent()
        self.report = ReportGeneratorAgent()
    def run(self, file_path):

        print("="*60)
        print("🚀 AI DATA ANALYST AGENT STARTED")
        print("="*60)

        # Step 1: Load Data
        df = self.loader.load_data(file_path)

        # Step 2: Understand Dataset
        understanding_report = self.understanding.analyze(df)

        # Step 3: Clean Data
        cleaned_df, cleaning_report = self.cleaner.clean_data(df)

        # Step 4: Detect Outliers
        outlier_report = self.outlier.detect_outliers(cleaned_df)

        # Step 5: EDA
        eda_report = self.eda.analyze(cleaned_df)

        # Step 6: Create Charts
        self.visualizer.create_visualization(cleaned_df)

        # Step 7: Generate Insights
        insights = self.insight.generate_insights(cleaned_df)
        
        cleaned_df.to_csv(
        "cleaned_employees.csv",
        index=False)

        print("✅ Cleaned dataset saved")

        # Step 8: Report Generation
        self.report.generate_report(
        understanding_report,
        cleaning_report,
        outlier_report,
        insights
        )
        print("\n✅ Analysis Completed Successfully")

        return {
            "cleaned_df": cleaned_df,
            "understanding_report": understanding_report,
            "cleaning_report": cleaning_report,
            "outlier_report": outlier_report,
            "eda_report": eda_report,
            "insights": insights
        }

# Run the Complete System

In [24]:
agent=AIAnalystAgent()
results=agent.run("employees.csv")

🚀 AI DATA ANALYST AGENT STARTED
✅ Dataset Loaded
✅ Data Cleaning Completed
✅ EDA Completed
✅ Visualizations Created
✅ Cleaned dataset saved
✅ Report Generated Successfully

✅ Analysis Completed Successfully
